# ProbGeo-UQ Proposal2 trên KITTI (Google Colab)

Notebook đọc ba archive KITTI trực tiếp từ
`MyDrive/KITTI_DATASET_ZIP`, giải nén vào ổ local Colab với thanh tiến trình,
prepare dữ liệu, train và đánh giá. Checkpoint/kết quả được lưu bền vững trên
Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

BRANCH = "Proposal2-Loss-Function"
%cd /content
!test -d Lidar || git clone https://github.com/danhyoyo/Lidar.git
%cd /content/Lidar
!git fetch origin "{BRANCH}:refs/remotes/origin/{BRANCH}"
!git checkout -B "{BRANCH}" "origin/{BRANCH}"
!git branch --show-current
!git log -1 --oneline

## Cấu hình

Chỉ cần đổi các giá trị trong cell dưới. `PHYSICAL_BATCH_SIZE` là nút chỉnh
chính nếu GPU thiếu bộ nhớ.

In [ ]:
from pathlib import Path
import os
import torch

REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/probgeo_uq_artifacts")

VARIANT = "B3"  # B0, B1, B2 hoặc B3
SEED = 42
PRECISION = "bf16"  # Cần GPU hỗ trợ BF16
PHYSICAL_BATCH_SIZE = 16
ACCUMULATION_STEPS = 1
NUM_WORKERS = max(0, min(8, (os.cpu_count() or 1) - 1))
EPOCHS = 100

## Cài thư viện và kiểm tra GPU

In [ ]:
%cd /content/Lidar
!python3 -m pip install -q shapely onnx tqdm
if _exit_code:
    raise RuntimeError("Không cài được Python dependencies")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

## Giải nén và prepare KITTI

Archive được stream trực tiếp từ Google Drive qua `tqdm` rồi vào `tar`, không tạo
bản copy local hàng chục GiB. Cell bỏ qua folder đã đủ số file.

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted_dir = RAW_KITTI_ROOT / "training" / folder
    extracted_count = sum(1 for _ in extracted_dir.glob(pattern))

    if extracted_count != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Không tìm thấy archive: {archive}")
        archive_bytes = archive.stat().st_size
        !set -o pipefail; python3 -m tqdm --bytes --total {archive_bytes} --desc "Giải nén {folder}" < "{archive}" | tar --no-same-owner -xf - -C "{RAW_KITTI_ROOT}"
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")

    actual_count = sum(1 for _ in extracted_dir.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: {actual_count} file, cần {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)

if not dataset_ready:
    %cd /content/Lidar
    !python3 tools/kitti_training_pipeline/prepare_kitti.py \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --output-root "{PROCESSED_DATASET_DIR}" \
      --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" \
      --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" \
      --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" \
      --pointcloud-mode symlink \
      --overwrite
    if _exit_code:
        raise RuntimeError("prepare_kitti.py thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
assert pointcloud_count == label_count == 7481
assert (PROCESSED_DATASET_DIR / "train.txt").is_file()
assert (PROCESSED_DATASET_DIR / "val.txt").is_file()

print(f"Raw KITTI: {RAW_KITTI_ROOT}")
print(f"Processed: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count}")

## Kiểm tra code

In [ ]:
EXPECTED_TESTS = {
    "legacy", "encoder", "shapes", "probgeo_config",
    "export_contract", "uncertainty_evaluator", "probgeo_review",
    "gates", "head", "targets", "loss", "gwd",
    "uncertainty_loss", "decode", "metrics", "selector",
    "training_guard",
}
TEST_LOG = Path("/tmp/mobilebev-test.log")

%cd /content/Lidar
!bash -o pipefail -c 'MPLCONFIGDIR=/tmp/mobilebev-mpl python3 tests/test_mobile_bev.py | tee /tmp/mobilebev-test.log'
test_exit_code = _exit_code
passed_tests = {
    line.removeprefix("PASS ").strip()
    for line in TEST_LOG.read_text(encoding="utf-8").splitlines()
    if line.startswith("PASS ")
}
if test_exit_code or passed_tests != EXPECTED_TESTS:
    raise RuntimeError(
        f"Test Proposal2 thất bại: exit_code={test_exit_code}, "
        f"missing={sorted(EXPECTED_TESTS - passed_tests)}, "
        f"unexpected={sorted(passed_tests - EXPECTED_TESTS)}"
    )
print("✅ Toàn bộ 17/17 kiểm tra đã PASS")

## Chọn variant và config

- `B0`: deterministic baseline.
- `B1`: deterministic + GWD.
- `B2`: heteroscedastic NLL.
- `B3`: heteroscedastic NLL + GWD.

In [ ]:
CONFIGS = {
    "B0": "configs/kitti/probgeo_uq/b0_deterministic.json",
    "B1": "configs/kitti/probgeo_uq/b1_gwd.json",
    "B2": "configs/kitti/probgeo_uq/b2_heteroscedastic.json",
    "B3": "configs/kitti/probgeo_uq/b3_probgeo_uq.json",
}

if VARIANT in CONFIGS:
    CONFIG = str(REPO_DIR / CONFIGS[VARIANT])
else:
    raise ValueError(f"VARIANT không hợp lệ: {VARIANT}")

EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS
RUN_NAME = f"probgeo_uq_{VARIANT.lower()}_seed{SEED}_b{EFFECTIVE_BATCH_SIZE}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Variant:", VARIANT)
print("Config:", CONFIG)
print("Run:", RUN_NAME)

## Smoke test

Chạy ngắn trước để phát hiện lỗi pipeline hoặc thiếu bộ nhớ.

In [ ]:
%cd /content/Lidar
!python3 tools/kitti_training_pipeline/train.py \
  --config "{CONFIG}" \
  --detector-root detector \
  --output-root "{ARTIFACT_ROOT}" \
  --run-name "{RUN_NAME}_smoke" \
  --device cuda \
  --precision "{PRECISION}" \
  --seed {SEED} \
  --epochs 1 \
  --physical-batch-size {PHYSICAL_BATCH_SIZE} \
  --accumulation-steps {ACCUMULATION_STEPS} \
  --max-train-batches 8 \
  --max-val-batches 4 \
  --num-workers {NUM_WORKERS}
if _exit_code:
    raise RuntimeError("Smoke test thất bại; giảm PHYSICAL_BATCH_SIZE nếu GPU hết bộ nhớ")

## Train full

In [ ]:
%cd /content/Lidar
import subprocess

RUN_DIR = ARTIFACT_ROOT / RUN_NAME
TRAIN_LOG = RUN_DIR / "train.log"
PID_PATH = RUN_DIR / "train.pid"
LAST_CHECKPOINT = RUN_DIR / "checkpoints" / "last.pt"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def is_training(pid):
    try:
        return b"tools/kitti_training_pipeline/train.py" in Path(f"/proc/{pid}/cmdline").read_bytes()
    except (FileNotFoundError, PermissionError):
        return False

TRAIN_PID = int(PID_PATH.read_text()) if PID_PATH.is_file() else 0
if not is_training(TRAIN_PID):
    candidates = list((RUN_DIR / "checkpoints").glob("*epoch.pt")) + list((RUN_DIR / "best_checkpoints").glob("*epoch.pt"))
    RESUME_CHECKPOINT = LAST_CHECKPOINT if LAST_CHECKPOINT.is_file() else max(candidates, key=lambda path: int(path.stem.removesuffix("epoch")), default=None)
    command = [
        "python3", "-u", "tools/kitti_training_pipeline/train.py",
        "--config", str(CONFIG), "--detector-root", "detector",
        "--output-root", str(ARTIFACT_ROOT), "--run-name", RUN_NAME,
        "--device", "cuda", "--precision", PRECISION,
        "--seed", str(SEED), "--epochs", str(EPOCHS),
        "--physical-batch-size", str(PHYSICAL_BATCH_SIZE),
        "--accumulation-steps", str(ACCUMULATION_STEPS),
        "--num-workers", str(NUM_WORKERS),
    ]
    if RESUME_CHECKPOINT:
        command += ["--resume", str(RESUME_CHECKPOINT)]
        print(f"Resume từ {RESUME_CHECKPOINT}")
    else:
        print(f"Train từ đầu đến epoch {EPOCHS}")
    with TRAIN_LOG.open("a") as stream:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    TRAIN_PID = process.pid
    PID_PATH.write_text(str(TRAIN_PID))
else:
    print(f"Training đang chạy với PID {TRAIN_PID}")

print(f"Theo dõi {TRAIN_LOG}; nếu cell hiện ^C, chạy lại cell này để nối log—training vẫn tiếp tục.")
!tail --pid={TRAIN_PID} -n 20 -f "{TRAIN_LOG}"

## Chọn checkpoint

B0–B3 chọn checkpoint theo 3D mAP trên 500 frame calibration.
Test split không được dùng trong bước này.

In [ ]:
%cd /content/Lidar
!python3 tools/kitti_training_pipeline/select_checkpoint.py \
  --checkpoint-dir "{ARTIFACT_ROOT / RUN_NAME / 'checkpoints'}" \
  --config "{CONFIG}" \
  --detector-root detector \
  --kitti-root "{RAW_KITTI_ROOT}" \
  --split splits/kitti/uq_calibration.txt \
  --output-dir "{ARTIFACT_ROOT / RUN_NAME / 'selected_3d'}" \
  --device cuda
if _exit_code:
    raise RuntimeError("Chọn checkpoint theo 3D mAP thất bại")
CHECKPOINT = ARTIFACT_ROOT / RUN_NAME / "selected_3d/best.pt"

if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
print("Checkpoint:", CHECKPOINT)

## Đánh giá

Đặt `RUN_EVALUATION = False` nếu chỉ cần train. Calibration và test
được đánh giá riêng; B2/B3 chạy thêm các metric uncertainty.

In [ ]:
%cd /content/Lidar
RUN_EVALUATION = True

if RUN_EVALUATION:
    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py \
      --name "{RUN_NAME}_calibration" \
      --backend pytorch \
      --model "{CHECKPOINT}" \
      --config "{CONFIG}" \
      --detector-root detector \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --split splits/kitti/uq_calibration.txt \
      --output "{ARTIFACT_ROOT / RUN_NAME / 'evaluation_calibration.json'}" \
      --device cuda \
      --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá calibration thất bại")

    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py \
      --name "{RUN_NAME}_test" \
      --backend pytorch \
      --model "{CHECKPOINT}" \
      --config "{CONFIG}" \
      --detector-root detector \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --split splits/kitti/uq_test.txt \
      --output "{ARTIFACT_ROOT / RUN_NAME / 'evaluation_test.json'}" \
      --device cuda \
      --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá test thất bại")

    if VARIANT in {"B2", "B3"}:
        !python3 tools/kitti_training_pipeline/evaluate_uncertainty.py \
          --predictions "{ARTIFACT_ROOT / RUN_NAME / 'evaluation_test.predictions.npz'}" \
          --config "{CONFIG}" \
          --kitti-root "{RAW_KITTI_ROOT}" \
          --split splits/kitti/uq_test.txt \
          --calibration-predictions "{ARTIFACT_ROOT / RUN_NAME / 'evaluation_calibration.predictions.npz'}" \
          --calibration-split splits/kitti/uq_calibration.txt \
          --output "{ARTIFACT_ROOT / RUN_NAME / 'uncertainty_test.json'}"
        if _exit_code:
            raise RuntimeError("Đánh giá uncertainty thất bại")